# Day 2: Parsing DataCite Metadata Step by Step

This notebook explains **how to read and parse a DataCite XML metadata record** for a teaching dataset.

## Learning goals
By the end of this notebook, students should be able to:
1. understand what a DataCite metadata record looks like,
2. identify the most important dataset fields,
3. parse the XML file step by step in Python,
4. convert the extracted metadata into a clean table,
5. connect DataCite metadata to the **FAIR principles**.

**Teaching dataset:** `2025 Global Climate Data`


In [1]:
# Notebook bootstrap: download required files from GitHub if missing
from pathlib import Path
from urllib.request import urlretrieve

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/MehrdadJalali-AI/data-management-teaching-materials/main"

required_files = [
    ("data/metadata/climate_dataset_datacite.xml", "data/metadata/climate_dataset_datacite.xml"),
]

for local_rel, github_rel in required_files:
    local_path = Path(local_rel)
    if not local_path.exists():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        url = f"{GITHUB_RAW_BASE}/{github_rel}"
        urlretrieve(url, local_path)
        print(f"Downloaded: {local_path}")
    else:
        print(f"Already available: {local_path}")

print("Bootstrap complete.")

Downloaded: data/metadata/climate_dataset_datacite.xml
Bootstrap complete.


## Step 1 — Import libraries

We use:
- `Path` to work with file paths,
- `xml.etree.ElementTree` to read XML,
- `pandas` to present the results clearly in table form.


In [2]:
from pathlib import Path
import xml.etree.ElementTree as ET
import pandas as pd

## Step 2 — Load the DataCite XML file

The metadata file is stored locally in the course repository under:

`data/metadata/climate_dataset_datacite.xml`


In [3]:
xml_path = Path("data/metadata/climate_dataset_datacite.xml")

print("XML path:", xml_path)
print("File exists:", xml_path.exists())

if not xml_path.exists():
    raise FileNotFoundError(f"Could not find: {xml_path}")

XML path: data/metadata/climate_dataset_datacite.xml
File exists: True


## Step 3 — Preview the raw XML

Before parsing, it is useful to look at the first part of the XML file.
This helps students see that metadata is stored as **structured tags**.


In [4]:
raw_xml = xml_path.read_text(encoding="utf-8")
print(raw_xml[:1500])

<?xml version="1.0" encoding="UTF-8"?>
<resource
  xmlns="http://datacite.org/schema/kernel-4"
  xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
  xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4/metadata.xsd">

  <identifier identifierType="DOI">10.1234/gcd-2025</identifier>

  <creators>
    <creator>
      <creatorName>Global Climate Data Team</creatorName>
    </creator>
  </creators>

  <titles>
    <title>2025 Global Climate Data</title>
  </titles>

  <publisher>Open Research Repository (Teaching)</publisher>

  <publicationYear>2025</publicationYear>

  <resourceType resourceTypeGeneral="Dataset">Dataset</resourceType>

  <descriptions>
    <description descriptionType="Abstract">
      A small teaching dataset containing fictional annual climate summaries.
      Variables include average temperature, rainfall, and CO2 emissions for two fictional countries
      (Alandia and Borealia) from 2021 to 2023.
    </description>
  

## Step 4 — Parse the XML document

Now we parse the XML into an element tree and inspect the root tag.


In [5]:
tree = ET.parse(xml_path)
root = tree.getroot()

print("Root tag:", root.tag)

Root tag: {http://datacite.org/schema/kernel-4}resource


## Step 5 — Understand XML namespaces

In XML metadata, tags often include a **namespace**.
For example, a tag may appear like:

`{http://datacite.org/schema/kernel-4}title`

For teaching, it is easier to remove the namespace and keep only the local tag name such as `title`.


In [6]:
def local_name(tag: str) -> str:
    """Return the tag name without the XML namespace."""
    return tag.split('}', 1)[1] if '}' in tag else tag

# Show a few example tags
example_tags = []
for el in root.iter():
    example_tags.append(el.tag)
    if len(example_tags) >= 10:
        break

for tag in example_tags:
    print("Original:", tag)
    print("Local   :", local_name(tag))
    print("---")

Original: {http://datacite.org/schema/kernel-4}resource
Local   : resource
---
Original: {http://datacite.org/schema/kernel-4}identifier
Local   : identifier
---
Original: {http://datacite.org/schema/kernel-4}creators
Local   : creators
---
Original: {http://datacite.org/schema/kernel-4}creator
Local   : creator
---
Original: {http://datacite.org/schema/kernel-4}creatorName
Local   : creatorName
---
Original: {http://datacite.org/schema/kernel-4}titles
Local   : titles
---
Original: {http://datacite.org/schema/kernel-4}title
Local   : title
---
Original: {http://datacite.org/schema/kernel-4}publisher
Local   : publisher
---
Original: {http://datacite.org/schema/kernel-4}publicationYear
Local   : publicationYear
---
Original: {http://datacite.org/schema/kernel-4}resourceType
Local   : resourceType
---


## Step 6 — Inspect the main tags in this DataCite record

Let us list the unique XML tag names used in this file.
This gives students a quick overview of the structure.


In [7]:
unique_tags = sorted({local_name(el.tag) for el in root.iter()})
pd.DataFrame({"tag_name": unique_tags})

,tag_name
0,creator
1,creatorName
2,creators
3,description
4,descriptions
5,identifier
6,publicationYear
7,publisher
8,resource
9,resourceType


## Step 7 — Create a simple parser function

We now build a small teaching-oriented parser to extract the most relevant DataCite fields:

- identifier
- creators
- title
- publisher
- publicationYear
- resourceType
- description


In [8]:
def parse_datacite_xml(xml_file: Path) -> dict:
    root = ET.parse(xml_file).getroot()

    identifier = None
    creators = []
    title = None
    publisher = None
    publication_year = None
    resource_type = None
    description = None

    for el in root.iter():
        name = local_name(el.tag)
        text = (el.text or "").strip()

        if name == "identifier" and text and identifier is None:
            identifier = text
        elif name == "creatorName" and text:
            creators.append(text)
        elif name == "title" and text and title is None:
            title = text
        elif name == "publisher" and text and publisher is None:
            publisher = text
        elif name == "publicationYear" and text and publication_year is None:
            publication_year = text
        elif name == "resourceType" and text and resource_type is None:
            resource_type = text
        elif name == "description" and text and description is None:
            description = text

    return {
        "identifier": identifier,
        "creators": creators,
        "title": title,
        "publisher": publisher,
        "publicationYear": publication_year,
        "resourceType": resource_type,
        "description": description,
    }

## Step 8 — Run the parser

Now we apply the parser to the local DataCite XML file.


In [9]:
datacite_meta = parse_datacite_xml(xml_path)
datacite_meta

{'identifier': '10.1234/gcd-2025',
 'creators': ['Global Climate Data Team'],
 'title': '2025 Global Climate Data',
 'publisher': 'Open Research Repository (Teaching)',
 'publicationYear': '2025',
 'resourceType': 'Dataset',
 'description': 'A small teaching dataset containing fictional annual climate summaries.\n      Variables include average temperature, rainfall, and CO2 emissions for two fictional countries\n      (Alandia and Borealia) from 2021 to 2023.'}

## Step 9 — Present the extracted metadata clearly

A dictionary is useful for Python, but a table is easier to discuss in class.


In [10]:
rows = [
    ("identifier", datacite_meta.get("identifier")),
    ("creators", "; ".join(datacite_meta.get("creators", []))),
    ("title", datacite_meta.get("title")),
    ("publisher", datacite_meta.get("publisher")),
    ("publicationYear", datacite_meta.get("publicationYear")),
    ("resourceType", datacite_meta.get("resourceType")),
    ("description", datacite_meta.get("description")),
]

table = pd.DataFrame(rows, columns=["field", "value"])
table

,field,value
0,identifier,10.1234/gcd-2025
1,creators,Global Climate Data Team
2,title,2025 Global Climate Data
3,publisher,Open Research Repository (Teaching)
4,publicationYear,2025
5,resourceType,Dataset
6,description,A small teaching dataset containing fictional ...


## Step 10 — Which fields are especially important for FAIR?

Now we interpret the metadata from a FAIR perspective.


In [11]:
fair_mapping = pd.DataFrame([
    ["identifier", "Findable", "A persistent identifier such as a DOI makes the dataset easier to locate and cite."],
    ["title", "Findable / Reusable", "A clear title helps users understand what the dataset is about."],
    ["creators", "Reusable", "Creator names support attribution and trust."],
    ["publisher", "Findable / Reusable", "Shows who published or manages the dataset."],
    ["publicationYear", "Reusable", "Provides temporal context for citation and interpretation."],
    ["resourceType", "Reusable", "Clarifies what kind of resource is being described."],
    ["description", "Accessible / Reusable", "Helps users understand the dataset before reusing it."],
], columns=["metadata_field", "supports_FAIR", "why_it_matters"])

fair_mapping

,metadata_field,supports_FAIR,why_it_matters
0,identifier,Findable,A persistent identifier such as a DOI makes th...
1,title,Findable / Reusable,A clear title helps users understand what the ...
2,creators,Reusable,Creator names support attribution and trust.
3,publisher,Findable / Reusable,Shows who published or manages the dataset.
4,publicationYear,Reusable,Provides temporal context for citation and int...
5,resourceType,Reusable,Clarifies what kind of resource is being descr...
6,description,Accessible / Reusable,Helps users understand the dataset before reus...


## Step 11 — Check metadata completeness

This is a simple classroom-friendly check, not a formal validator.


In [12]:
required_fields = ["identifier", "creators", "title", "publisher", "publicationYear", "resourceType", "description"]

completeness_rows = []
for field in required_fields:
    value = datacite_meta.get(field)
    if isinstance(value, list):
        present = len(value) > 0
    else:
        present = value is not None and str(value).strip() != ""
    completeness_rows.append((field, present))

completeness_df = pd.DataFrame(completeness_rows, columns=["field", "present"])
completeness_df["score"] = completeness_df["present"].astype(int)

completeness_df

,field,present,score
0,identifier,True,1
1,creators,True,1
2,title,True,1
3,publisher,True,1
4,publicationYear,True,1
5,resourceType,True,1
6,description,True,1


In [13]:
completeness_score = completeness_df["score"].sum()
max_score = len(completeness_df)
print(f"Metadata completeness score: {completeness_score}/{max_score}")

Metadata completeness score: 7/7


## Step 12 — Teaching interpretation

### Strengths of this DataCite record
- It includes a clear **identifier**.
- It records **creator names**.
- It provides a dataset **title**.
- It includes **publisher** and **publication year**.
- It contains a short **description**.

### Possible weaknesses or limitations
- The description may still be too short for full reuse.
- Technical details such as format, license, or variable-level documentation may be missing.
- DataCite is strong for dataset citation, but not always sufficient alone for full domain-specific description.

### Key teaching message
DataCite helps make a dataset more **Findable** and **Reusable**, especially when citation and persistent identification are important.


## Short class discussion questions

1. Which DataCite field is most important for **Findability**?
2. Why are creator names useful for **Reuse**?
3. Is a title alone enough for understanding a dataset?
4. What additional metadata would improve reuse in practice?
5. How is DataCite different from Dublin Core or schema.org?
